In [1]:
# 基于LangChain的大语言模型应用--基于文档问答
# 目的：使用LLM回答关于提供的文档的问题
# 从本章节开始会开始设计Embedding模型和向量存储(Vector Stores)
# embedding模型会把一段文字翻译成一段向量，意思越相近，向量值越接近
# 注意deepseek本身不是embedding模型，因此在此处使用的是阿里云的模型
# ollama pull ryanshillington/Qwen3-Embedding-8B
import os
api_key = os.environ.get("DEEPSEEK_API_KEY")
qwen_api_key = os.environ.get("QWEN_API_KEY")

In [38]:
# 导入构建链必备的库
from langchain_classic.chains import RetrievalQA # 帮助检索文档
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.document_loaders import CSVLoader # 文件加载器
from langchain_classic.vectorstores import DocArrayInMemorySearch # 向量存储，使用内存进行存储
from IPython.display import display, Markdown

In [3]:
llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash"
)

In [4]:
file = "OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file,encoding="utf-8")

In [7]:
# 导入索引，帮助创建向量存储
from langchain_classic.indexes import  VectorstoreIndexCreator
from langchain_openai import OpenAIEmbeddings

In [41]:
embedding = OpenAIEmbeddings(
    api_key=qwen_api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen3.7-text-embedding",
    check_embedding_ctx_length=False, # 发送原始文本
    # DashScope单词最多运行20条
    chunk_size=20
)

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch, # 此处的向量存储方式可以进行替换
    embedding=embedding
).from_loaders([loader]) # 调用文档加载器，传入包含加载器的列表

In [42]:
query = "Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

In [14]:
response = index.query(query, llm=llm)

In [15]:
response

"| Name | Summary |\n|------|---------|\n| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable, wrinkle-free, quick-drying shirt with UPF 50+ protection (blocks 98% UV rays). Made from 52% polyester/48% nylon, machine washable, with front/back cape venting and two bellows pockets. Designed for hot weather and travel. |\n| Sun Shield Shirt | High-performance, slightly fitted sun shirt with UPF 50+ (SPF 50+) protection. Made from 78% nylon/22% Lycra Xtra Life fiber for abrasion resistance and moisture-wicking comfort. Handwash and line dry; fits over swimsuits; recommended by The Skin Cancer Foundation. |\n| Men's TropicVibe Shirt, Short-Sleeve | Traditional fit, lightweight sun-protection shirt with UPF 50+, made from 71% nylon/29% polyester with a polyester mesh lining. Wrinkle resistant, features cape venting and two front bellows pockets; machine washable and dryable. |\n| Men's Tropical Plaid Short-Sleeve Shirt | Lightest hot-weather shirt with UPF 50+ protection. 100% polyest

In [40]:
display(Markdown(response))

| Name | Summary |
|------|---------|
| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable, wrinkle-free, quick-drying shirt with UPF 50+ protection (blocks 98% UV rays). Made from 52% polyester/48% nylon, machine washable, with front/back cape venting and two bellows pockets. Designed for hot weather and travel. |
| Sun Shield Shirt | High-performance, slightly fitted sun shirt with UPF 50+ (SPF 50+) protection. Made from 78% nylon/22% Lycra Xtra Life fiber for abrasion resistance and moisture-wicking comfort. Handwash and line dry; fits over swimsuits; recommended by The Skin Cancer Foundation. |
| Men's TropicVibe Shirt, Short-Sleeve | Traditional fit, lightweight sun-protection shirt with UPF 50+, made from 71% nylon/29% polyester with a polyester mesh lining. Wrinkle resistant, features cape venting and two front bellows pockets; machine washable and dryable. |
| Men's Tropical Plaid Short-Sleeve Shirt | Lightest hot-weather shirt with UPF 50+ protection. 100% polyester, wrinkle-resistant, traditional fit, with front/back cape venting and two front bellows pockets. Machine washable; highest rated sun protection possible. |

In [44]:
# 语言模型一次性只能接收有限个单词，如果存在一个较大的文档，
# 此时需要embedding和向量数据库的帮助，Embedding将一段文字转换成数字，用一段数字表示这段文本，其中，相似意义的文本具有相似的向量值，因此可以在向量空间中比较文本片段，这种技术可以找出跟问题相似的文本片段，一起传递给语言模型来帮助回答问题
# 向量数据库可以存储前文创建的矢量数字数组，向向量数据库中新建数据，就是把文档拆分为块，每块生成Embedding，把生成的Embedding和原始块一起存储到数据库当中
# 在处理较大的文本时，首先需要将文本分为几个小块，每次只需要把最相关的几块内容传递给语言模型，然后根据每个文本块生成一个Embedding，将其存储在向量数据当中 -> 创建索引
# 当需要查询时，首先为传进来的query生成一个Embedding，得到一个数字数组，将该数组与向量数据库中的所有向量进行比较，选择最相似的若干个文本块
# 找到目标文本块后，将这些文本块和原始的查询内容一起传递给语言模型，让语言模型根据文档内容生成答案

In [56]:
# 对底层操作进行研究
from langchain_classic.document_loaders import CSVLoader
loader = CSVLoader(file_path=file,encoding="utf-8") # 加载文档，注意加载编码

In [57]:
docs = loader.load()

In [59]:
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [80]:
# 由于此时文本并不大，因此可以无需将文本分块，直接生成Embedding
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
        api_key=qwen_api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen3.7-text-embedding",
    check_embedding_ctx_length=False, # 发送原始文本
    # DashScope单词最多运行20条
    chunk_size=20 # 此处为调用模型的问题
)

In [82]:
embed = embeddings.embed_query("Hi my name is Hilary")

In [84]:
print(len(embed))

1024


In [86]:
print(embed[:5])

[0.010084624402225018, -0.01859372854232788, -0.05687493458390236, 0.003971349913626909, -0.004700515419244766]


In [87]:
# 为刚才加载的所有文本片段生成Embedding，并存储在一个向量存储器当中
db = DocArrayInMemorySearch.from_documents(docs, embeddings)

In [89]:
query = "Please suggest a shirt with sunblocking"

In [90]:
docs = db.similarity_search(query)

In [92]:
len(docs)

4

In [95]:
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 255}, page_content=': 255\nname: Sun Shield Shirt by\ndescription: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. \n\nSize & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.\n\nFabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.\n\nAdditional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.\n\nSun Protection That Won\'t Wear Off\nOur high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun\'s harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.')

In [96]:
# 回答自己文档当中的问题
retriever = db.as_retriever() #  定义了一个接收查询内容并且返回相似文档的方法

In [97]:
llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash",
    temperature=0.0
) # 希望使用自然语言返回

In [99]:
qdocs = "".join([docs[i].page_content for i in range(len(docs))])

In [103]:
# 将变量当中的内容和问题传给语言大模型
response = llm.invoke(f"{qdocs} Question: Please list all your shirt with sun protection in a table with markdown and summarize each one").content

In [105]:
display(Markdown(response))

Here are the sun-protection shirts from the provided list. All are rated UPF 50+, which blocks 98% of harmful UV rays.

| # | Shirt Name | Summary |
|---|---|---|
| 1 | **Sun Shield Shirt** | Women’s high-performance sun shirt with a slightly fitted, hip-length silhouette. Made from 78% nylon / 22% Lycra Xtra Life fiber. UPF 50+ rated, handwash and line dry. Wicks moisture, fits comfortably over swimsuits, and is abrasion resistant. |
| 2 | **Men's Plaid Tropic Shirt, Short-Sleeve** | Men’s short-sleeve hot-weather shirt originally designed for fishing. UPF 50+ with SunSmart technology, wrinkle-free, and quick-drying. Made from 52% polyester / 48% nylon, machine washable and dryable. Features cape venting and two front bellows pockets. |
| 3 | **Sunrise Tee** | Women’s UV-protective button-down shirt that is lightweight, moisture-wicking, quick-drying, and wrinkle-free. UPF 50+ rated. Shell is 71% nylon / 29% polyester with 100% polyester cape lining. Machine wash and dry. Includes cape venting, two front pockets, tool tabs, eyewear loop, and side shaping. |
| 4 | **Girls' Ocean Breeze Long-Sleeve Stripe Shirt** | Girls’ long-sleeve sun-protection rash guard with full coverage. UPF 50+ rated nylon Lycra®-elastane blend. Quick-drying, fade-resistant, machine wash and line dry. Holds shape well, resists seawater damage, and coordinates with swimsuits. |

In short: all four shirts offer **UPF 50+ sun protection**, are made with performance fabrics, and are designed for warm-weather activities like beach days, fishing, and travel.

In [107]:
# 使用LangChain chain来链接所有的步骤
# 创建一个RetrievalQA链
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True
)

In [108]:
query = "Please list all your shirt with sun protection in a table with markdown and summarize each one"

In [111]:
response = qa_stuff.run(query)



> Entering new RetrievalQA chain...

> Finished chain.


In [113]:
display(Markdown(response)) # 打印Markdown

Here are all the shirts with sun protection from the provided information:

| Name | Summary |
|------|---------|
| Sun Shield Shirt | High-performance sun shirt with UPF 50+ protection, blocking 98% of harmful UV rays. Slightly fitted, falls at hip. Made of 78% nylon / 22% Lycra Xtra Life fiber. Wicks moisture, abrasion resistant, and fits over swimsuits. Handwash, line dry. |
| Men's Plaid Tropic Shirt, Short-Sleeve | Lightweight hot-weather shirt rated UPF 50+ with SunSmart technology blocking 98% of UV rays. Wrinkle-free, quick-drying, and breathable with front/back cape venting and two bellows pockets. Made of 52% polyester / 48% nylon. Machine washable and dryable. |
| Sunrise Tee | Women’s UV-protective button-down shirt rated UPF 50+. Made of lightweight performance fabric (71% nylon / 29% polyester) that wicks moisture, resists wrinkles, and dries fast. Features cape venting, two front pockets, tool tabs, and eyewear loop. Machine wash and dry. |
| Men's TropicVibe Shirt, Short-Sleeve | Men’s short-sleeve sun-protection shirt with UPF 50+ rating. Traditional fit, wrinkle resistant, with front/back cape venting and two bellows pockets. Shell: 71% nylon / 29% polyester; knit mesh lining. Machine wash and dry. |

All four shirts offer UPF 50+ sun protection, which blocks 98% of harmful UV rays and is the highest sun protection rating available.

In [115]:
# 上述为工作的震整个流程
response = index.query(query, llm=llm)

In [117]:
display(Markdown(response))

| Product | Sun Protection | Summary |
|---------|---------------|---------|
| Sun Shield Shirt by | UPF 50+ (SPF 50+) | Slightly fitted, falls at hip. Made of 78% nylon / 22% Lycra Xtra Life fiber. Wicks moisture, abrasion resistant, handwash/line dry, fits over swimsuits. Blocks 98% of UV rays. |
| Men's Plaid Tropic Shirt, Short-Sleeve | UPF 50+ | Lightweight hot-weather shirt originally designed for fishing. Wrinkle-free, quick-drying, with front/back cape venting and two front bellows pockets. Made of 52% polyester / 48% nylon; machine washable and dryable. Blocks 98% of UV rays. |
| Men's TropicVibe Shirt, Short-Sleeve | UPF 50+ | Traditional relaxed fit. Shell: 71% nylon / 29% polyester with polyester knit mesh lining. Wrinkle resistant, cape venting, two front bellows pockets. Machine wash/dry. Blocks 98% of UV rays. |
| Men's Tropical Plaid Short-Sleeve Shirt | UPF 50+ | Lightest hot-weather shirt with traditional relaxed fit. 100% polyester, wrinkle-resistant, front/back cape venting, two front bellows pockets. Blocks 98% of UV rays. |

In [118]:
# stuff方法只需要把所有的内容放到Prompt当中，然后发送给语言模型，就能得到返回结果

+ Map_reduce: 对所有的分块，把每一块的内容连同问题一起传递给语言模型，得到一个独立返回的结果，每一块得到的结果都合并在一起，再使用语言模型来对这些结果进行总结
    + 特点：需要调用更多的语言模型，而且它把所有文档都独立处理，不能够总是最理想的结果
+ Refine：迭代进行，基于前一个文档的答案，适合需要整合信息，以及随着时间推移构建答案非常有用
    + 会导致更长的答案，速度没有那么快，每一个文档无法独立调用，必须依赖前面的结果
+ Map_rerank：对于每个文档，只需对语言模型进行一次调用，还需要额外返回一个评分，然后选择最高分的结果，依赖于语言模型对分数的判定
    + 所有调用独立，响应速度更快，但是需要多次调用语言模型